# 3D-VGA — step 1: text → speech

Synthesises one WAV per ScanRefer description with Piper: six voices, of which the last two are heard at validation only. Built for **Run All**; it resumes where it stopped.

Writes `data_audio/wav/<scene>/<object>_<ann>.wav` (16 kHz mono) and `data_audio/manifest_<split>.json` to the Colab local disk (`/content`), then runs the frozen speech encoder over every WAV (`data_audio/feat/<encoder>/<split>.h5`), and finally zips `data/`, `cached_scenes/`, `outputs/` and `data_audio/` (WAVs, manifests, features) into one archive copied to Google Drive. Next: `run_audio_train_eval.ipynb`. Next: `run_audio_encode.ipynb`.

## 0 — download data

Fetches one archive from Google Drive that holds `data/`, `cached_scenes/` and `outputs/`, and extracts it next to this notebook. Skipped when those three folders are already on disk.

In [6]:
# DOWNLOAD DATA  (Google Drive -> zip/rar -> extract)
# One archive ("files") holding data/, cached_scenes/ and outputs/ is fetched and extracted next to this
# notebook. It is skipped on later runs if all three folders already exist. Set the link to "" to skip.

LINKS = {
    "files": "https://drive.google.com/file/d/1dM5rlP2u22f2iApQKUiBBi1r1QEazTDf/view?usp=sharing",   # https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing
}
EXPECTED_FOLDERS = ["data", "cached_scenes", "outputs"]   # what the archive must contain at its top level
FORCE_REDOWNLOAD = False   # True: download and extract again even if the folder exists

import subprocess, sys, shutil, zipfile
from pathlib import Path

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

def extract(archive, dest):
    archive = Path(archive)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
    elif archive.suffix.lower() == ".rar":
        if shutil.which("unrar") is None and shutil.which("unar") is None and shutil.which("7z") is None:
            subprocess.call(["apt-get", "install", "-y", "-q", "unrar"])
        if shutil.which("unrar"):
            subprocess.check_call(["unrar", "x", "-o+", "-inul", str(archive), str(dest) + "/"])
        elif shutil.which("unar"):
            subprocess.check_call(["unar", "-q", "-f", "-o", str(dest), str(archive)])
        elif shutil.which("7z"):
            subprocess.check_call(["7z", "x", "-y", f"-o{dest}", str(archive)])
        else:
            raise RuntimeError("no rar extractor available (install unrar, unar or 7z)")
    else:
        raise RuntimeError(f"{archive.name}: not a zip or rar archive")

def folders_ready():
    return all(Path(f).is_dir() and any(Path(f).iterdir()) for f in EXPECTED_FOLDERS)

for name, link in LINKS.items():
    if not link:
        print(f"[skip] {name}: no link given")
        continue
    if folders_ready() and not FORCE_REDOWNLOAD:
        print(f"[skip] {', '.join(EXPECTED_FOLDERS)} already exist")
        continue
    archive = Path(f"{name}_download.archive")
    print(f"[get ] {name} ...")
    out = gdown.download(url=link, output=str(archive), quiet=False, fuzzy=True)
    if out is None:
        print(f"[fail] {name}: download failed (check the link is shared as 'anyone with the link')")
        continue
    print(f"[unpack] {name} ...")
    extract(archive, ".")      # the archive is expected to contain data/, cached_scenes/ and outputs/ at its top level
    archive.unlink()
    for f in EXPECTED_FOLDERS:
        print(f"[done] {f}/ -> {'ok' if Path(f).is_dir() else 'NOT FOUND after extract: archive layout differs'}")


[skip] data/ already exists
[skip] cached_scenes/ already exists
[skip] outputs/ already exists


In [7]:
import shutil
from pathlib import Path

#-------------------------------------------------------------------------------
!git clone https://github.com/neo-the-hammer/3DVGA.git
#-------------------------------------------------------------------------------

source_dir = Path('/content/3DVGA')
target_dir = Path('/content')

if source_dir.exists() and source_dir.is_dir():
    for item in source_dir.iterdir():
        dest = target_dir / item.name
        if dest.exists():
            if dest.is_dir():
                shutil.rmtree(dest)
            else:
                dest.unlink()
        shutil.move(str(item), str(target_dir))
    source_dir.rmdir()
    print("Successfully moved all contents of 3DVGA to the main folder and removed the empty 3DVGA directory.")
else:
    print("3DVGA folder not found.")

Cloning into '3DVGA'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 130 (delta 16), reused 130 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 275.19 KiB | 15.29 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Successfully moved all contents of 3DVGA to the main folder and removed the empty 3DVGA directory.


## 1 — configuration

The only cell you need to edit.

In [8]:
# USER CONFIGURATION

DEVICE     = "cuda"      # "auto" | "cpu" | "cuda"   -- auto probes for a working GPU
GPU_ID     = "0"
QUICK_TEST = False       # True: 2 scenes per split into data_audio/_quick -- minutes, and
                         #       never mixed with the real corpus

SPLITS = ["train", "val"]

RUN_TTS     = True
TTS_BACKEND = "piper"    # piper | coqui | espeak | gtts
VOICES = [               # piper voice ids; missing models are downloaded automatically
    "en_US-amy-medium", "en_US-ryan-medium", "en_GB-jenny_dioco-medium", "en_US-joe-medium",
    "en_US-lessac-medium", "en_GB-alan-medium",     # <- the last HELD_OUT_VOICES: val only
]
HELD_OUT_VOICES = 2      # the last N voices never speak a training utterance (speaker-OOD test)
TTS_ON_GPU      = "auto" # True | False | "auto" -- needs onnxruntime-gpu; falls back to CPU
TTS_INSTANCES   = 1      # CPU synthesis in N parallel processes (ignored on GPU)
OVERWRITE_WAVS  = False  # False resumes: WAVs that exist are kept

FRAME_CAP_SECONDS = 10.24 # what the encoding step keeps of each clip (256 frames at 25/s)

# GOOGLE DRIVE -- Colab only

MIRROR_TO_DRIVE = True
DRIVE_FOLDER    = "3DVGA"        # under MyDrive/
WAVS_ON_DRIVE   = False          # False: WAVs and features stay on the Colab local disk (/content); step 6d zips
                                 #        everything and copies the zip to Drive. True: WAVs are written to Drive directly
AUDIO_ZIP_NAME  = "audio_data"   # -> /content/audio_data.zip, copied to MyDrive/<DRIVE_FOLDER>/

QUICK_SCENES = 2

# ENCODING (step 6b)

RUN_ENCODE        = True
ENCODER           = "wav2vec2"         # wav2vec2 | wav2vec2-large | wavlm | hubert
ENCODER_LAYER     = -1                 # hidden layer read; -1 = the encoder's default
                                       # (8 of 12; 16 of 24 for wav2vec2-large)
MAX_AUDIO_LEN     = 256                # frames kept per utterance; 256 at 25 frames/s = 10.24 s
ENCODE_BATCH_SIZE = 8                  # a batch that does not fit the GPU is retried clip by clip

## 2 — environment

Probes the GPU by launching a kernel — `torch.cuda.is_available()` alone can report `True` on an unsupported card.

In [9]:
import glob as _glob
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

REPO = Path.cwd()
RESULTS = {}

if not (REPO / "scripts" / "audio_train.py").is_file():
    raise SystemExit(f"Run this notebook from the repository root. Current cwd: {REPO}")

# lib/config.py lists this directory at import time; an empty one is enough for the steps
# that never read a scan.
(REPO / "data" / "scannet" / "scans").mkdir(parents=True, exist_ok=True)

IN_COLAB = importlib.util.find_spec("google.colab") is not None

GPU_USABLE, GPU_NAME, GPU_NOTE = False, None, ""
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        try:
            _probe = torch.randn(8, 8, device="cuda")
            (_probe @ _probe).sum().item()
            GPU_USABLE = True
            GPU_NOTE = f"{GPU_NAME} (sm_{major}{minor})"
        except Exception as error:
            arch = getattr(torch.cuda, "get_arch_list", lambda: ["?"])()
            GPU_NOTE = (f"{GPU_NAME} (sm_{major}{minor}) present but NO KERNEL RUNS -- "
                        f"this torch ships {arch}. "
                        f"{type(error).__name__}: {str(error).splitlines()[0][:80]}")
    else:
        GPU_NOTE = "no CUDA device visible to torch"
except Exception as error:
    GPU_NOTE = f"torch unavailable: {error}"

if DEVICE == "auto":
    DEVICE = "cuda" if GPU_USABLE else "cpu"
elif DEVICE == "cuda" and not GPU_USABLE:
    print("[warn] DEVICE='cuda' but no kernel launches here.")

print("=" * 88)
print(f"environment  : {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"python       : {sys.version.split()[0]}")
print(f"GPU          : {'USABLE -- ' + GPU_NOTE if GPU_USABLE else 'UNUSABLE -- ' + GPU_NOTE}")
print(f"device       : {DEVICE}")
print(f"quick test   : {QUICK_TEST}")
print("=" * 88)

environment  : Google Colab
python       : 3.13.16
GPU          : USABLE -- Tesla T4 (sm_75)
device       : cuda
quick test   : False


### Runner

Streams a command's output and keeps its log under `outputs/reports/<step>/`.

In [10]:
REPORTS = REPO / "outputs" / "reports"

def run(key, cmd, env=None, note=""):
    """Run one command, stream its output, and keep a report under outputs/reports/."""
    cmd = [str(c) for c in cmd]
    printable = " ".join(cmd)
    print("=" * 88)
    print(f"STEP   {key}" + (f"\n       {note}" if note else ""))
    print(f"$ {printable}\n{'-' * 88}")

    full_env = os.environ.copy()
    full_env.update({k: str(v) for k, v in (env or {}).items()})
    full_env["CUDA_VISIBLE_DEVICES"] = GPU_ID if (DEVICE == "cuda" and GPU_USABLE) else ""
    full_env["PYTHONUNBUFFERED"] = "1"

    started = time.time()
    captured = []
    process = subprocess.Popen(cmd, cwd=str(REPO), env=full_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        captured.append(line)
    code = process.wait()
    elapsed = round(time.time() - started, 1)

    folder = REPORTS / key
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "log.txt").write_text("".join(captured))
    record = {"key": key, "note": note, "cmd": printable, "exit_code": code,
              "elapsed_sec": elapsed, "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
              "environment": "Google Colab" if IN_COLAB else "local", "gpu": GPU_NAME or "none"}
    (folder / "result.json").write_text(json.dumps(record, indent=2))
    RESULTS[key] = record

    if code in (247, -9, 137):
        verdict = f"FAILED (exit {code}) -- killed by the OS, out of host RAM"
    else:
        verdict = "OK" if code == 0 else f"FAILED (exit {code})"
    print(f"{'-' * 88}\n[{key}] {verdict} in {elapsed}s   log -> {folder.relative_to(REPO)}/log.txt")
    return code

## 3 — packages

Colab only. `onnxruntime-gpu` replaces the CPU build so Piper can synthesise on the GPU. If the GPU cannot run it, synthesis falls back to the CPU by itself.

In [11]:
EXTRA_PACKAGES = ["piper-tts", "soundfile", "transformers", "h5py"]
REQUIRED_MODULES = ["torch", "numpy", "piper", "onnxruntime", "easydict", "h5py", "transformers"]

def _pip(*args):
    print("$ pip install " + " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

def ensure_packages(force=False):
    if not (IN_COLAB or force):
        print("not on Colab -- skipping installs. Pass force=True to run them anyway.")
        return
    if (REPO / "req" / "requirements.txt").is_file():
        _pip("-r", str(REPO / "req" / "requirements.txt"))
    _pip(*EXTRA_PACKAGES)

ensure_packages()

print("\n-- packages --")
MISSING_PACKAGES = []
for module in REQUIRED_MODULES:
    try:
        loaded = importlib.import_module(module)
        print(f"  {module:18s} OK    {getattr(loaded, '__version__', '')}")
    except Exception:
        MISSING_PACKAGES.append(module)
        print(f"  {module:18s} --    MISSING")
if MISSING_PACKAGES:
    print(f"\n  install first: pip install {' '.join(MISSING_PACKAGES)}")

def _onnx_ok():
    """Import onnxruntime in a fresh process; returns (importable, provider list)."""
    probe = subprocess.run(
        [sys.executable, "-c",
         "from onnxruntime import InferenceSession, get_available_providers; print(get_available_providers())"],
        capture_output=True, text=True)
    return probe.returncode == 0, probe.stdout

def _reinstall_onnx(*packages):
    # Uninstall BOTH builds first: leftovers of either leave an empty `onnxruntime` namespace
    # package ("cannot import name InferenceSession ... unknown location").
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                    "onnxruntime", "onnxruntime-gpu"], check=False)
    _pip("--force-reinstall", "--no-deps", *packages)

if IN_COLAB and TTS_BACKEND == "piper":
    if GPU_USABLE and TTS_ON_GPU:
        _reinstall_onnx("onnxruntime-gpu")
    ok, _ = _onnx_ok()
    if not ok:
        print("[warn] onnxruntime is broken after the GPU build; installing the CPU build instead.")
        _reinstall_onnx("onnxruntime")
        ok, _ = _onnx_ok()
    print("onnxruntime importable:", ok)

TTS_GPU = False
if TTS_ON_GPU and GPU_USABLE and TTS_BACKEND == "piper":
    ok, providers = _onnx_ok()
    TTS_GPU = ok and "CUDAExecutionProvider" in providers
print(f"\nspeech will be synthesised on: {'GPU' if TTS_GPU else 'CPU'}")


$ pip install -r /content/req/requirements.txt
$ pip install piper-tts soundfile

-- packages --
  torch              OK    2.11.0+cu130
  numpy              OK    2.1.3
  piper              OK    
  onnxruntime        OK    1.31.0
  easydict           OK    
$ pip install onnxruntime-gpu

speech will be synthesised on: GPU


## 4 — Google Drive and paths

On Colab the WAVs are written to the local disk (`/content/data_audio/`); the zip made in step 6b is what goes to Drive. Locally everything stays under `data_audio/`.

In [12]:
# Off Colab this whole cell is inert, so the notebook stays runnable locally.

DRIVE_ROOT = None

def mount_drive():
    """Mount Drive and return the destination directory, or None if unavailable."""
    global DRIVE_ROOT
    if not (MIRROR_TO_DRIVE and IN_COLAB):
        print("Drive off"
              + ("" if MIRROR_TO_DRIVE else " (MIRROR_TO_DRIVE=False)")
              + ("" if IN_COLAB else " -- not running on Colab, everything stays local"))
        return None
    try:
        from google.colab import drive
        mount = Path("/content/drive")
        if not (mount / "MyDrive").is_dir():
            drive.mount(str(mount))
        DRIVE_ROOT = mount / "MyDrive" / DRIVE_FOLDER
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted -> {DRIVE_ROOT}")
    except Exception as error:
        # A failed mount must not take the run with it.
        DRIVE_ROOT = None
        print(f"Drive not mounted ({error}). The run continues; results stay on this "
              f"machine only and will be lost when the runtime recycles.")
    return DRIVE_ROOT

def restore_audio_zip():
    """Fresh runtime: unpack data_audio/ (WAVs, manifests, features) from the zip kept on Drive."""
    import zipfile
    if DRIVE_ROOT is None or WAVS_ON_DRIVE or QUICK_TEST:
        return
    drive_zip = DRIVE_ROOT / f"{AUDIO_ZIP_NAME}.zip"
    if drive_zip.is_file() and not (REPO / "data_audio" / "wav").exists():
        print(f"restoring {drive_zip.name} from Drive ...")
        with zipfile.ZipFile(drive_zip) as z:
            members = [n for n in z.namelist() if n.startswith("data_audio/")]
            z.extractall(REPO, members)   # only data_audio/; data, cached_scenes, outputs stay as they are

def copy_newer(source, target, quiet=False):
    """Copy a file or a tree, skipping files the target already has up to date."""
    source, target = Path(source), Path(target)
    if not source.exists():
        return 0
    files = [source] if source.is_file() else [p for p in source.rglob("*") if p.is_file()]
    copied = 0
    for path in files:
        if "tensorboard" in path.parts or path.suffix == ".tmp":
            continue
        dest = target if source.is_file() else target / path.relative_to(source)
        try:
            if dest.exists() and dest.stat().st_mtime >= path.stat().st_mtime \
                    and dest.stat().st_size == path.stat().st_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, dest)
            copied += 1
        except Exception as error:
            if not quiet:
                print(f"  could not copy {path.name}: {error}")
    return copied

mount_drive()
restore_audio_zip()

LOCAL_ROOT = REPO / "data_audio" / ("_quick" if QUICK_TEST else "")
if DRIVE_ROOT is not None and WAVS_ON_DRIVE and not QUICK_TEST:
    AUDIO_ROOT = DRIVE_ROOT / "data_audio"
else:
    AUDIO_ROOT = LOCAL_ROOT          # Colab local disk: /content/data_audio
FEAT_DIR = LOCAL_ROOT / "feat"           # always local: training reads it at every step; zipped at the end

DATA_ENV = {
    "VGA_AUDIO_ROOT": AUDIO_ROOT,
    "VGA_AUDIO_FEAT_DIR": FEAT_DIR,
    "VGA_HELD_OUT_VOICES": HELD_OUT_VOICES,
    "VGA_ENCODER_LAYER": ENCODER_LAYER,
    "VGA_MAX_AUDIO_LEN": MAX_AUDIO_LEN,
}
SUBSET = ["--num_scenes", QUICK_SCENES] if QUICK_TEST else []

print(f"WAVs and manifests : {AUDIO_ROOT}")

Mounted at /content/drive
Drive mounted -> /content/drive/MyDrive/3DVGA
WAVs and manifests : /content/drive/MyDrive/3DVGA/data_audio


## 5 — preflight

What is on disk and what is left to do. Changes nothing.

In [13]:
def annotations(split):
    data = json.loads((REPO / "data" / f"ScanRefer_filtered_{split}.json").read_text())
    if QUICK_TEST:
        scenes = set(sorted({d["scene_id"] for d in data})[:QUICK_SCENES])
        data = [d for d in data if d["scene_id"] in scenes]
    return data

def manifest(split):
    path = AUDIO_ROOT / f"manifest_{split}.json"
    return json.loads(path.read_text()) if path.is_file() else None

def cache_path(split):
    return FEAT_DIR / ENCODER / f"{split}.h5"

def cached_keys(split):
    if not cache_path(split).is_file():
        return set()
    import h5py
    with h5py.File(cache_path(split), "r") as handle:
        return {k.decode("utf-8") for k in handle["keys"][:]}

def key_of(d):
    return f"{d['scene_id']}/{d['object_id']}/{d['ann_id']}"

print(f"{'split':<7} {'descriptions':>13} {'WAVs (manifest)':>16} {'in feature cache':>17}")
print("-" * 58)
to_speak = 0
to_encode = 0
for split in SPLITS:
    wanted = {key_of(d) for d in annotations(split)}
    listed = manifest(split)
    spoken = {e["key"] for e in (listed or {}).get("entries", []) if "error" not in e} & wanted
    encoded = cached_keys(split) & wanted
    to_speak += len(wanted) - len(spoken)
    to_encode += len(wanted) - len(encoded)
    print(f"{split:<7} {len(wanted):>13} {len(spoken):>16} {len(encoded):>17}")

print(f"\n{to_speak} clip(s) to synthesise; {to_encode} to encode.")
if to_speak and not TTS_GPU:
    print("CPU synthesis takes roughly 0.3-1 s per clip depending on the cores available; "
          "the full corpus is hours.\nIt resumes, so an interrupted session loses nothing.")

split    descriptions  WAVs (manifest)
----------------------------------------
train           36665                0
val              9508                0

46173 clip(s) to synthesise.


## 5b — compare TTS engines and choose one

Optional. Picks hard sentences (long, digits, abbreviations, odd punctuation, rare words, plus a few fixed stress tests), then lets you synthesise each with any engine, listen, rate it, and finally **set the engine for the whole corpus**. Run it before step 6; "Set as final" overwrites `TTS_BACKEND` and `VOICES`, which step 6 reads.

Engines: `piper` (installed), `espeak` (installs espeak-ng), `gtts` (needs internet), `coqui` (heavy, optional `pip install TTS`).

In [ ]:
# 5b - TTS ENGINE LAB
import re, importlib.util, io, numpy as np
import ipywidgets as W
from IPython.display import Audio, display, clear_output

N_HARD = 6                      # hard ScanRefer sentences to pick
ENGINES = ["piper", "espeak", "gtts", "coqui"]

STRESS = [
    "The 2 chairs, next to the tv-stand (about 3 ft high), face the L-shaped sofa.",
    "It is the 3rd door on the left of the big brown cabinet; not the white one!",
    "This is a small, round, wooden table that sits between two black office chairs.",
    "There's a trash-can under the desk, beside the 'Dell' monitor and the PC tower.",
]

def _load_builder():
    spec = importlib.util.spec_from_file_location("build_audioset", REPO / "scripts/audio_tts/build_audioset.py")
    sys.path.insert(0, str(REPO))
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod
builder = _load_builder()

def hardness(t):
    w = t.split()
    return (len(w) / 10 + 2 * len(re.findall(r"\d", t)) + t.count(",") + t.count(";") + t.count("(")
            + 2 * len(re.findall(r"\b[a-z]{1,3}[-.][a-z]+\b|\b[A-Z]{2,}\b", t))
            + sum(len(x) > 9 for x in w) * 0.5)

pool = []
for _s in SPLITS:
    pool += [d["description"] for d in annotations(_s)]
hard = sorted(set(builder.normalise_text(t) for t in pool if t), key=hardness, reverse=True)
TEXTS = STRESS + [t for t in hard if len(t) < 400][:N_HARD]

_cache = {}   # engine -> backend instance
def get_backend(name):
    if name not in _cache:
        if name == "espeak" and shutil.which("espeak-ng") is None:
            subprocess.run(["apt-get", "install", "-y", "-q", "espeak-ng"], check=False)
        if name == "gtts" and importlib.util.find_spec("gtts") is None:
            _pip("gTTS", "soundfile")
        if name == "coqui" and importlib.util.find_spec("TTS") is None:
            _pip("TTS")
        kw = dict(voice_list=VOICES if name == "piper" else None, cuda=bool(TTS_GPU))
        _cache[name] = builder.BACKENDS[name](**kw)
    return _cache[name]

RATINGS = {}   # (engine, voice) -> 1..5
engine_dd = W.Dropdown(options=ENGINES, value=TTS_BACKEND if TTS_BACKEND in ENGINES else "piper", description="engine")
voice_dd  = W.Dropdown(description="voice")
text_dd   = W.Dropdown(options=[(f"{i+1}. {t[:70]}", t) for i, t in enumerate(TEXTS)], description="text")
text_box  = W.Textarea(value=TEXTS[0], layout=W.Layout(width="95%", height="70px"), description="custom")
rate_sl   = W.IntSlider(value=3, min=1, max=5, description="quality")
out       = W.Output()

def refresh_voices(*_):
    out.clear_output()
    try:
        voice_dd.options = get_backend(engine_dd.value).voices()
    except BaseException as e:
        voice_dd.options = []
        with out: print(f"{engine_dd.value} unavailable: {e}")
engine_dd.observe(refresh_voices, "value")
text_dd.observe(lambda c: setattr(text_box, "value", c["new"]), "value")

def speak(_=None):
    with out:
        clear_output()
        try:
            be = get_backend(engine_dd.value)
            audio, rate = be.synthesize(text_box.value, voice_dd.value)
            print(f"{engine_dd.value} / {voice_dd.value}: {len(audio)/rate:.2f} s @ {rate} Hz")
            display(Audio(np.asarray(audio), rate=rate))
        except BaseException as e:
            print("synthesis failed:", type(e).__name__, e)

def rate_it(_=None):
    RATINGS[(engine_dd.value, voice_dd.value)] = rate_sl.value
    with out: print("ratings:", {f"{k[0]}/{k[1]}": v for k, v in RATINGS.items()})

def sweep(_=None):
    """Same text, first voice of every engine, one after another."""
    with out:
        clear_output()
        for name in ENGINES:
            try:
                be = get_backend(name); v = be.voices()[0]
                audio, rate = be.synthesize(text_box.value, v)
                print(f"\n{name} / {v}  ({len(audio)/rate:.2f} s)")
                display(Audio(np.asarray(audio), rate=rate))
            except BaseException as e:
                print(f"\n{name}: skipped ({type(e).__name__}: {str(e)[:80]})")

def set_final(_=None):
    global TTS_BACKEND, VOICES, TTS_GPU
    TTS_BACKEND = engine_dd.value
    if TTS_BACKEND == "piper":
        keep = [v for v in VOICES]               # keep the configured piper voices
        VOICES = keep
    else:
        TTS_GPU = False
    with out:
        print(f"FINAL engine set: TTS_BACKEND={TTS_BACKEND!r}"
              + (f", VOICES={VOICES}" if TTS_BACKEND == "piper" else ", voices come from the engine")
              + "\nRun step 6 to synthesise everything with it. (Set OVERWRITE_WAVS=True if WAVs from another engine exist.)")

btns = {}
for label, fn in [("Synthesize", speak), ("Rate", rate_it), ("Compare all engines", sweep), ("Set as final", set_final)]:
    b = W.Button(description=label, button_style="success" if label == "Set as final" else "")
    b.on_click(fn); btns[label] = b

refresh_voices()
display(W.VBox([W.HBox([engine_dd, voice_dd]), text_dd, text_box, W.HBox([rate_sl]), W.HBox(list(btns.values())), out]))


## 6 — synthesise

The voice is a fixed function of the annotation, so re-running assigns the same voices. With `TTS_INSTANCES > 1` several CPU processes share the work.

In [14]:
# switch: RUN_TTS
if RUN_TTS:
    cmd = [sys.executable, "scripts/audio_tts/build_audioset.py", "--splits", *SPLITS,
           "--backend", TTS_BACKEND, *SUBSET]
    cmd = [str(c) for c in cmd]
    if TTS_BACKEND == "piper":
        cmd += ["--voice_list", *VOICES]
    if OVERWRITE_WAVS:
        cmd += ["--overwrite"]

    if TTS_GPU:
        cmd += ["--cuda"]
    elif TTS_INSTANCES > 1:
        # Each instance takes every N-th description and keeps its own manifest; the
        # ordinary run below then finds every WAV in place and merges the manifests.
        env = os.environ.copy()
        env.update({k: str(v) for k, v in DATA_ENV.items()})
        (REPORTS / "audio-tts").mkdir(parents=True, exist_ok=True)
        workers = []
        for i in range(1, TTS_INSTANCES + 1):
            log = open(REPORTS / "audio-tts" / f"instance_{i}.log", "w")
            workers.append(subprocess.Popen([*cmd, "--shard", f"{i}/{TTS_INSTANCES}"],
                                            cwd=str(REPO), env=env, stdout=log,
                                            stderr=subprocess.STDOUT))
        print(f"{TTS_INSTANCES} instances running; logs in outputs/reports/audio-tts/")
        codes = [w.wait() for w in workers]
        print("instance exit codes:", codes)

    run("audio-tts", cmd, env=DATA_ENV, note="ScanRefer descriptions -> WAVs")
    # Later notebooks read the manifests from the local tree.
    for split in SPLITS:
        copy_newer(AUDIO_ROOT / f"manifest_{split}.json", LOCAL_ROOT / f"manifest_{split}.json")
else:
    print("RUN_TTS is off")

STEP   audio-tts
       ScanRefer descriptions -> WAVs
$ /usr/bin/python3 scripts/audio_tts/build_audioset.py --splits train val --backend piper --voice_list en_US-amy-medium en_US-ryan-medium en_GB-jenny_dioco-medium en_US-joe-medium en_US-lessac-medium en_GB-alan-medium --cuda
----------------------------------------------------------------------------------------
[3D-VGA] environment overrides: HELD_OUT_VOICES=2
[piper] downloading voice en_US-amy-medium -> /content/data_audio/voices
[piper] downloading voice en_US-ryan-medium -> /content/data_audio/voices
[piper] downloading voice en_GB-jenny_dioco-medium -> /content/data_audio/voices
[piper] downloading voice en_US-joe-medium -> /content/data_audio/voices
[piper] downloading voice en_US-lessac-medium -> /content/data_audio/voices
[piper] downloading voice en_GB-alan-medium -> /content/data_audio/voices
[train] 36665 annotations, 6 voice(s), backend=piper
2026-10-24 11:55:19.267277666 [W:onnxruntime:, transformer_memcpy.cc:111 Appl

## 6b — encode

Runs the frozen speech encoder once over every WAV and stores its frames at 25 per second, with six prosody channels, in `data_audio/feat/<encoder>/<split>.h5`. Written to a temporary file and renamed at the end, so an interrupted run leaves no half-built cache. Needs a GPU for the full corpus.

In [ ]:
# switch: RUN_ENCODE
if RUN_ENCODE:
    cmd = [sys.executable, "scripts/audio_encode/cache_audio_features.py", "--splits", *SPLITS,
           "--encoder", ENCODER, "--layer", ENCODER_LAYER,
           "--batch_size", ENCODE_BATCH_SIZE, *SUBSET]
    run("audio-encode", cmd, env=DATA_ENV, note="WAVs -> frozen encoder features")
    # No separate copy to Drive: the features are zipped and sent in step 6d.
else:
    print("RUN_ENCODE is off")


## 6c — verify

Every description should have features. An utterance missing from the cache is heard as silence in training, so this must read `READY` before the final zip.

In [ ]:
import numpy as np
import h5py

ready = True
for split in SPLITS:
    path = cache_path(split)
    if not path.is_file():
        print(f"{split}: no feature cache at {path}")
        ready = False
        continue
    wanted = {key_of(d) for d in annotations(split)}
    with h5py.File(path, "r") as handle:
        keys = {k.decode("utf-8") for k in handle["keys"][:]}
        lengths = np.diff(handle["offsets"][:])
        attrs = dict(handle.attrs)
        prosody = handle["prosody"][: min(20000, handle["prosody"].shape[0])]
    missing = wanted - keys
    ready &= not missing
    print(f"\n{split}: {path.name}  {path.stat().st_size / 1e9:.2f} GB")
    print(f"  encoder {attrs['encoder']} layer {attrs['encoder_layer']}, "
          f"{attrs['fps']:.0f} frames/s, dim {attrs['feat_dim']}, cap {attrs['max_audio_len']}")
    print(f"  utterances {len(keys & wanted)} of {len(wanted)}   missing {len(missing)}")
    print(f"  frames per utterance: mean {lengths.mean():.1f}  max {lengths.max()}  "
          f"at the cap {100 * (lengths >= attrs['max_audio_len']).mean():.1f}%")
    print(f"  prosody channel std: {np.round(prosody.std(axis=0), 2)}  (a zero means a dead channel)")

print("\n" + ("READY -- continue with the final zip below" if ready
              else "NOT READY -- some descriptions have no features; re-run the stages above"))


## 6d — zip everything and copy to Drive

One archive holding `data/`, `cached_scenes/`, `outputs/` and `data_audio/` (WAVs, manifests, encoded features). Built on the Colab local disk, then copied to `MyDrive/<DRIVE_FOLDER>/`.

In [16]:
# 6d - ZIP data, cached_scenes, outputs, data_audio (WAVs + features) and TRANSFER the zip to Google Drive
import shutil, zipfile
from pathlib import Path

do_copy = True

if do_copy:
    if not globals().get("ready", False):
        print("[warn] step 6c did not report READY -- the archive will hold an incomplete corpus.")
    if AUDIO_ROOT != LOCAL_ROOT:
        print("[note] WAVS_ON_DRIVE is on: the WAVs are on Drive already and are not part of this zip.")
    folders_to_zip = ["data", "cached_scenes", "outputs", "data_audio"]
    local_zip_path = REPO / f"{AUDIO_ZIP_NAME}.zip"           # /content/audio_data.zip

    print(f"Creating {local_zip_path.name} ...")
    with zipfile.ZipFile(local_zip_path, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        for folder in folders_to_zip:
            src = REPO / folder
            if not src.exists():
                print(f"Warning: {folder} does not exist, skipped.")
                continue
            print(f"Adding {folder}/ ...")
            for path in sorted(src.rglob("*")):
                if path.is_file():
                    heavy = path.suffix in (".wav", ".h5", ".pth", ".npy")    # already dense: store, do not deflate
                    z.write(path, path.relative_to(REPO).as_posix(),
                            compress_type=zipfile.ZIP_STORED if heavy else zipfile.ZIP_DEFLATED)

    size_mb = local_zip_path.stat().st_size / (1024 * 1024)
    print(f"Archive created: {local_zip_path} ({size_mb:.2f} MB)")

    if DRIVE_ROOT is not None and DRIVE_ROOT.exists():
        drive_zip_path = DRIVE_ROOT / local_zip_path.name      # MyDrive/3DVGA/audio_data.zip
        print(f"Copying archive to Google Drive: {drive_zip_path} ...")
        shutil.copy2(local_zip_path, drive_zip_path)
        print("Successfully copied archive to Google Drive!")
    else:
        print("Google Drive is not mounted -- the zip stays on the local disk only.")


Creating zip archive...
Adding cached_scenes...
Adding data...
Adding outputs...
Archive created: files.zip (6017.84 MB)
Copying archive to Google Drive: /content/drive/MyDrive/3DVGA/files.zip...
Successfully copied archive to Google Drive!


## 7 — listen

Three random clips with their text and voice, clip lengths, and how many clips each voice spoke.

In [15]:
import random
import numpy as np
from IPython.display import Audio, display

listed = manifest("val") or manifest(SPLITS[0])
if listed is None:
    print("no manifest yet -- run stage 1")
else:
    spoken = [e for e in listed["entries"] if "duration" in e]
    for entry in random.Random(0).sample(spoken, min(3, len(spoken))):
        scene, obj, ann = entry["key"].split("/")
        print(f"\n{entry['key']}   voice={entry['voice']}   {entry['duration']} s\n  {entry['text']}")
        display(Audio(str(AUDIO_ROOT / "wav" / scene / f"{obj}_{ann}.wav")))

    durations = np.array([e["duration"] for e in spoken])
    cap = FRAME_CAP_SECONDS
    print(f"\n{len(spoken)} clip(s), {len(listed['voices'])} voice(s)")
    print(f"duration  mean {durations.mean():.2f} s   median {np.median(durations):.2f} s   "
          f"p95 {np.percentile(durations, 95):.2f} s")
    print(f"{100 * (durations > cap).mean():.1f}% are longer than the {cap:.2f} s cap "
          f"the encoding step keeps, and will be centre-cropped")
    voices = {}
    for entry in spoken:
        voices[entry["voice"]] = voices.get(entry["voice"], 0) + 1
    print("clips per voice:", voices)
    print("\nNext: run_audio_train_eval.ipynb")


scene0583_00/2/1   voice=en_GB-alan-medium   6.165 s
  This is a white porcelain toilet with a plastic seat. this object is between the sink and the window.



scene0608_00/28/4   voice=en_US-ryan-medium   3.727 s
  It is the dark sofa chair in the living room. it sits in front of a center table.



scene0077_00/2/3   voice=en_US-joe-medium   7.407 s
  A printer is placed in the middle of a cabinet top that is attached to the far wall. just above it are some shelves along with window slightly to the left.



9508 clip(s), 6 voice(s)
duration  mean 4.63 s   median 4.23 s   p95 8.23 s
1.8% are longer than the 10.24 s cap the encoding step keeps, and will be centre-cropped
clips per voice: {'en_US-lessac-medium': 1588, 'en_US-joe-medium': 1628, 'en_GB-alan-medium': 1534, 'en_US-ryan-medium': 1593, 'en_GB-jenny_dioco-medium': 1599, 'en_US-amy-medium': 1566}

Next: run_audio_encode.ipynb
